# Training Amount Effect Analysis

**Research Question (c):** How did the amount of training data used affect the tokens learnt?

## Comparisons

### For PI1M Concatenated (clean comparison across training amounts):
- **1 epoch (68M)** vs **5 epoch (240M)** vs **22 epoch (1B)**: `PI1M_concat_1epoch` vs `PI1M_concat_5epoch` vs `PI1M_concat_22epoch`

This provides a clean comparison across 1, 5, and 22 epochs all with the same concatenation strategy.

## Analysis Goals
1. Determine if more training data significantly changes tokenization
2. Look for token stability/convergence
3. Identify potential overfitting indicators


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
sns.set_context("talk")
colors = sns.color_palette("mako", 10)
plt.rcParams['figure.figsize'] = (14, 8)

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from analysis.utils.statistics import TokenStatistics, compare_token_distributions, compute_kl_divergence

# Load statistics
stats_dir = project_root / 'analysis' / 'data' / 'statistics'
pi1m_concat_1epoch = TokenStatistics.load(str(stats_dir / 'PI1M_concat_1epoch_stats.json'))
pi1m_concat_5epoch = TokenStatistics.load(str(stats_dir / 'PI1M_concat_5epoch_stats.json'))
pi1m_concat_22epoch = TokenStatistics.load(str(stats_dir / 'PI1M_concat_22epoch_stats.json'))

print("✓ Statistics loaded!")


/opt/pytorch/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


✓ Statistics loaded!


## 1. Concatenated: 1 epoch vs 5 epoch


In [2]:
comparison_1v5 = compare_token_distributions(
    pi1m_concat_1epoch, pi1m_concat_5epoch,
    label1="1 epoch (68M)", label2="5 epoch (240M)"
)

print("="*60)
print("PI1M Concatenated: 1 epoch vs 5 epoch")
print("="*60)
print(f"Token Overlap Jaccard: {comparison_1v5['token_overlap']['jaccard_similarity']:.4f}")
print(f"Breakpoint Overlap Jaccard: {comparison_1v5['breakpoint_overlap']['jaccard_similarity']:.4f}")
print(f"Mean Token Length Diff: {comparison_1v5['length_comparison']['mean_diff']:.4f}")

kl_div_1v5 = compute_kl_divergence(pi1m_concat_1epoch, pi1m_concat_5epoch)
print(f"KL Divergence: {kl_div_1v5:.4f}")


PI1M Concatenated: 1 epoch vs 5 epoch
Token Overlap Jaccard: 0.3889
Breakpoint Overlap Jaccard: 0.8222
Mean Token Length Diff: 0.4109
KL Divergence: 1.9468


## 2. Concatenated: 5 epoch vs 22 epoch


In [3]:
comparison_5v22 = compare_token_distributions(
    pi1m_concat_5epoch, pi1m_concat_22epoch,
    label1="5 epoch (240M)", label2="22 epoch (1B)"
)

print("="*60)
print("PI1M Concatenated: 5 epoch vs 22 epoch")
print("="*60)
print(f"Token Overlap Jaccard: {comparison_5v22['token_overlap']['jaccard_similarity']:.4f}")
print(f"Breakpoint Overlap Jaccard: {comparison_5v22['breakpoint_overlap']['jaccard_similarity']:.4f}")
print(f"Mean Token Length Diff: {comparison_5v22['length_comparison']['mean_diff']:.4f}")

kl_div_5v22 = compute_kl_divergence(pi1m_concat_5epoch, pi1m_concat_22epoch)
print(f"KL Divergence: {kl_div_5v22:.4f}")


PI1M Concatenated: 5 epoch vs 22 epoch
Token Overlap Jaccard: 0.3514
Breakpoint Overlap Jaccard: 0.8636
Mean Token Length Diff: 0.2581
KL Divergence: 1.9030


## 3. Concatenated: 1 epoch vs 22 epoch


In [4]:
comparison_1v22 = compare_token_distributions(
    pi1m_concat_1epoch, pi1m_concat_22epoch,
    label1="1 epoch (68M)", label2="22 epoch (1B)"
)

print("="*60)
print("PI1M Concatenated: 1 epoch vs 22 epoch")
print("="*60)
print(f"Token Overlap Jaccard: {comparison_1v22['token_overlap']['jaccard_similarity']:.4f}")
print(f"Breakpoint Overlap Jaccard: {comparison_1v22['breakpoint_overlap']['jaccard_similarity']:.4f}")
print(f"Mean Token Length Diff: {comparison_1v22['length_comparison']['mean_diff']:.4f}")

kl_div_1v22 = compute_kl_divergence(pi1m_concat_1epoch, pi1m_concat_22epoch)
print(f"KL Divergence: {kl_div_1v22:.4f}")


PI1M Concatenated: 1 epoch vs 22 epoch
Token Overlap Jaccard: 0.3333
Breakpoint Overlap Jaccard: 0.9130
Mean Token Length Diff: 0.6690
KL Divergence: 2.0447


## 4. Summary: Training Amount Effect


In [5]:
summary_data = {
    'Metric': [
        'Token Overlap Jaccard',
        'Breakpoint Overlap Jaccard',
        'Mean Token Length Diff',
        'KL Divergence',
    ],
    'Concat (1→5 epoch)': [
        f"{comparison_1v5['token_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_1v5['breakpoint_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_1v5['length_comparison']['mean_diff']:.4f}",
        f"{kl_div_1v5:.4f}",
    ],
    'Concat (5→22 epoch)': [
        f"{comparison_5v22['token_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_5v22['breakpoint_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_5v22['length_comparison']['mean_diff']:.4f}",
        f"{kl_div_5v22:.4f}",
    ],
    'Concat (1→22 epoch)': [
        f"{comparison_1v22['token_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_1v22['breakpoint_overlap']['jaccard_similarity']:.4f}",
        f"{comparison_1v22['length_comparison']['mean_diff']:.4f}",
        f"{kl_div_1v22:.4f}",
    ],
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*60)
print("Training Amount Effect Summary (All Concatenated PI1M)")
print("="*60)
print(summary_df.to_string(index=False))

print("\n" + "="*60)
print("Interpretation:")
print("="*60)
print("Higher Jaccard = more stable tokenization (converged)")
print("Lower KL divergence = less change with more training")
print("\nThis clean comparison (all concatenated) shows how tokenization")
print("evolves from 1 to 5 to 22 epochs with the same concatenation strategy.")

summary_df.to_csv(project_root / 'analysis' / 'data' / 'training_amount_summary.csv', index=False)
print("\n✓ Analysis complete!")



Training Amount Effect Summary (All Concatenated PI1M)
                    Metric Concat (1→5 epoch) Concat (5→22 epoch) Concat (1→22 epoch)
     Token Overlap Jaccard             0.3889              0.3514              0.3333
Breakpoint Overlap Jaccard             0.8222              0.8636              0.9130
    Mean Token Length Diff             0.4109              0.2581              0.6690
             KL Divergence             1.9468              1.9030              2.0447

Interpretation:
Higher Jaccard = more stable tokenization (converged)
Lower KL divergence = less change with more training

This clean comparison (all concatenated) shows how tokenization
evolves from 1 to 5 to 22 epochs with the same concatenation strategy.

✓ Analysis complete!
